# 2. Embeddings

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
v1.dot(dv)

np.float32(0.32332397)

In [4]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.019730445)

3. Embedding Dataset

In [7]:
from src import FaqHttpLoader

loader = FaqHttpLoader()
documents = loader.load()
print(f"Loaded {len(documents)} documents")

Loaded 1208 documents


In [11]:
documents[10]

{'question': 'Do I need to enroll in the course before submitting homework?',
 'text': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.',
 'section': 'General Course-Related Questions',
 'course': 'machine-learning-zoomcamp'}

Generating embeddings

In [12]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['text']
    texts.append(text)

In [15]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/25 [00:00<?, ?it/s]

1208

In [21]:
import numpy as np
X = np.array(vectors)
print(X.shape)

(1208, 384)


# 4. Vector Search

In [40]:
query = 'Can I still join the course after the start date?'
v_query = model.encode(query)

In [41]:
scores = X.dot(v_query)
# Best match
idx = np.argmax(scores)

print(f"Q: {query}")
print(f"Document [score={scores[idx]:.2%}]:\n{documents[idx]}")

Q: Can I still join the course after the start date?
Document [score=76.29%]:
{'question': 'Course: Can I still join the course after the start date?', 'text': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'section': 'General Course-Related Questions', 'course': 'data-engineering-zoomcamp'}


Top 5 results

In [48]:
top5 = np.argsort(-scores)[:5]
top5, scores[top5].round(3)

(array([553, 955,  29, 472, 558]),
 array([0.763, 0.758, 0.719, 0.654, 0.56 ], dtype=float32))

In [49]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'question': 'Course: Can I still join the course after the start date?', 'text': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'section': 'General Course-Related Questions', 'course': 'data-engineering-zoomcamp'}

0.7579371
{'question': 'Course - Can I still join the course after the start date?', 'text': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'section': 'General Course-Related Questions', 'course': 'mlops-zoomcamp'}

0.71921307
{'question': 'The course has already started. Can I still join it?', 'text': 'Yes, you can. Even though you missed the start date, you can

# 5. Vector Search with minsearch